# NB02: Growth Kinetics Analysis

**Project**: CF Protective Microbiome Formulation Design  
**Goal**: Extract growth parameters (lag time, max growth rate, carrying capacity) from fitted growth curves. These kinetic parameters are expected to predict competitive outcomes better than endpoint OD alone — a commensal that grows *faster* on a shared substrate depletes it before PA14 can ramp up.

## Sections
1. Load and characterize growth curve data
2. Visualize growth curves by isolate and condition
3. Fit growth models and extract kinetic parameters
4. Compute kinetic advantage scores vs PA14
5. Compare kinetics to endpoint OD — do they add information?

**Input**: `~/protect/gold/fact_growth_curves_fitted.snappy.parquet`, `fact_growth_curves_clean.snappy.parquet`, `fact_growth_curve.snappy.parquet`  
**Output**: `data/growth_parameters.tsv` (isolate × condition × {lag, rate, capacity})

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

GOLD = Path.home() / 'protect' / 'gold'
DATA = Path('..') / 'data'
FIGS = Path('..') / 'figures'
DATA.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print('Ready.')

## 1. Load and Characterize Growth Curve Data

Three growth curve tables exist at different processing stages:
- `fact_growth_curve` (17K rows): Summary/metadata per isolate × condition
- `fact_growth_curves_clean` (267K rows): Cleaned time-series OD
- `fact_growth_curves_fitted` (676K rows): Fitted OD values across ~193 time cycles

We use the fitted curves for parameter extraction.

In [ ]:
gc_fitted = pd.read_parquet(GOLD / 'fact_growth_curves_fitted.snappy.parquet')
gc_clean = pd.read_parquet(GOLD / 'fact_growth_curves_clean.snappy.parquet')
gc_summary = pd.read_parquet(GOLD / 'fact_growth_curve.snappy.parquet')
isolates = pd.read_parquet(GOLD / 'dim_isolate.snappy.parquet')

print(f'Fitted curves: {len(gc_fitted)} rows')
print(f'  Isolates: {gc_fitted.asma_id.nunique()}')
print(f'  Columns: {list(gc_fitted.columns)}')
print(f'\nClean curves: {len(gc_clean)} rows')
print(f'  Columns: {list(gc_clean.columns)}')
print(f'\nSummary: {len(gc_summary)} rows')
print(f'  Columns: {list(gc_summary.columns)}')

gc_fitted.head()

In [ ]:
# Understand the structure
print('=== Fitted curve structure ===')
print(f'Isolates: {sorted(gc_fitted.asma_id.unique())}')
print(f'\nAssays/conditions: {gc_fitted.columns.tolist()}')

# Check if 'condition' or 'assay' column exists, or if conditions are encoded differently
for col in gc_fitted.columns:
    print(f'\n{col}: dtype={gc_fitted[col].dtype}, nunique={gc_fitted[col].nunique()}, sample={gc_fitted[col].iloc[:3].tolist()}')

In [ ]:
# Same for clean and summary
print('=== Clean curve structure ===')
for col in gc_clean.columns:
    print(f'{col}: dtype={gc_clean[col].dtype}, nunique={gc_clean[col].nunique()}, sample={gc_clean[col].iloc[:3].tolist()}')

print('\n=== Summary structure ===')
for col in gc_summary.columns:
    print(f'{col}: dtype={gc_summary[col].dtype}, nunique={gc_summary[col].nunique()}, sample={gc_summary[col].iloc[:3].tolist()}')

In [ ]:
# Merge isolate taxonomy
gc_tax = gc_fitted.merge(isolates[['asma_id', 'species', 'genus']], on='asma_id', how='left')
print(f'\nIsolates with growth curves by species:')
sp = gc_tax.drop_duplicates('asma_id').species.value_counts()
print(sp.to_string())

## 2. Visualize Growth Curves

Show representative curves for PA14 and candidate commensals across carbon sources. This lets the audience see the raw kinetic data driving the analysis.

In [ ]:
# Plot growth curves — adapt based on actual data structure discovered above
# This cell will be refined after cell 2 reveals the exact column structure

# Placeholder: assuming columns are asma_id, assay/condition, cycle, od_fit
# Will be updated after EDA
print('Growth curve visualization will be built after structure is confirmed in cells above.')
print('See cell outputs above for column names and structure.')

## 3. Extract Growth Parameters

For each isolate × carbon source combination, extract:
- **Lag time (λ)**: Time before exponential growth begins
- **Max growth rate (μ_max)**: Steepest slope of the growth curve
- **Carrying capacity (K)**: Maximum OD reached

Method: fit a modified Gompertz model or extract directly from the fitted curves.

In [ ]:
def gompertz(t, A, mu, lam, y0):
    """Modified Gompertz growth model.
    A = carrying capacity (asymptote - y0)
    mu = max specific growth rate
    lam = lag time
    y0 = initial OD
    """
    return y0 + A * np.exp(-np.exp((mu * np.e / A) * (lam - t) + 1))


def extract_growth_params_numerical(times, od_values):
    """Extract growth parameters directly from OD time series.
    Numerical approach — no model fitting, works on any curve shape.
    """
    od = np.array(od_values, dtype=float)
    t = np.array(times, dtype=float)
    
    if len(od) < 5 or np.all(np.isnan(od)):
        return {'lag': np.nan, 'mu_max': np.nan, 'capacity': np.nan}
    
    # Carrying capacity: max OD
    capacity = np.nanmax(od)
    y0 = od[0] if not np.isnan(od[0]) else np.nanmin(od)
    
    # Growth rate: max derivative (smoothed)
    if len(od) >= 7:
        od_smooth = savgol_filter(od, min(7, len(od)//2*2+1), 2)
    else:
        od_smooth = od
    dt = np.diff(t)
    dt[dt == 0] = 1e-6
    dod_dt = np.diff(od_smooth) / dt
    mu_max = np.nanmax(dod_dt)
    
    # Lag time: time when growth rate first exceeds 10% of max rate
    threshold = 0.1 * mu_max if mu_max > 0 else 0
    lag_idx = np.where(dod_dt > threshold)[0]
    lag = t[lag_idx[0]] if len(lag_idx) > 0 else t[-1]
    
    return {'lag': lag, 'mu_max': mu_max, 'capacity': capacity}


print('Growth parameter extraction functions defined.')

In [ ]:
# Extract parameters for each isolate × condition
# NOTE: This cell will be adapted based on the actual data structure
# discovered in Section 1 above

print('Parameter extraction will proceed after growth curve structure is confirmed.')
print('The extract_growth_params_numerical() function handles arbitrary time series.')

## 4. Kinetic Advantage Scores

For each commensal isolate and each of PA14's preferred substrates, compute:
- **Rate advantage**: commensal μ_max / PA14 μ_max (>1 = commensal grows faster)
- **Lag advantage**: PA14 lag - commensal lag (>0 = commensal starts growing first)
- **Overall kinetic score**: combines rate and lag advantages

In [ ]:
# Will be computed after parameter extraction in Section 3
print('Kinetic advantage scores will be computed after growth parameters are extracted.')

## 5. Kinetics vs Endpoint OD — Do They Add Information?

Compare growth kinetic features (μ_max, lag) with simple endpoint OD from `fact_carbon_utilization`. If kinetics are highly correlated with endpoint OD, they may not add much. If they capture different aspects (e.g., fast grower with low yield, or slow grower with high yield), they're valuable additional features.

In [ ]:
# Will be computed after Sections 3-4
print('Kinetics vs endpoint comparison will follow parameter extraction.')